# Capstone, mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yashalaf/flyrank-internship/blob/main/work/notebooks/capstone.ipynb)

This skeleton is yours to fill. Work the sections **in order**, each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Research question:** Out of thousands of live content pages, which ones should an editor prioritize for refresh review this week, given a realistic capacity of about 50 reviews?

**Decision this supports:** a content editor's weekly refresh-priority queue. The cost of a wrong call isn't abstract: a false positive wastes a review slot on a page that's fine, a false negative lets a real decliner keep losing traffic silently while its slot goes to something else.

This question was settled in ML-02/ML-03 by checking signal strength across four possible framings (ranking-position analysis, content-type clustering, CTR opportunity scoring, and refresh/decline scoring) before committing to one, not by assumption.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/yashalaf/flyrank-internship"
REPO_DIR = "flyrank-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("Lane: Refresh / Content Opportunity Scoring")
print("Task shape: Ranking/Scoring, built on a binary classification target")
print("Decision: weekly refresh-priority queue, ~50 review slots/week")
print(f"Rows: {len(df)} | Clients: {df['client_id'].nunique()} | Declining share: {df['is_declining_label'].mean():.3f}")

Lane: Refresh / Content Opportunity Scoring
Task shape: Ranking/Scoring, built on a binary classification target
Decision: weekly refresh-priority queue, ~50 review slots/week
Rows: 30000 | Clients: 32 | Declining share: 0.542


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Release actually used, stated plainly:** `data/raw/content_refresh_anonymized.csv`, the starter/sample release (30,000 rows, 32 pseudonymized clients, 44 columns), not the full ~79M-row production warehouse.

**Honest note on scope:** ML-04 explored the full warehouse directly (`fact_content_daily_performance_sample.parquet`, 11,694,072 rows for June 2026 alone) via a gated Hugging Face connection, and found real, load-bearing issues there: duplicate rows for one client on one date, and a documented `access_profile`/availability-flag mechanism gating which columns are real measurements versus unmeasured. That exploration is real and is referenced in Methodology and Limitations, but the modeling pipeline in this paper (baseline, leakage audit, model, validation, action playbook) was built and validated on the smaller starter release, since sustained warehouse access wasn't available in the environment used to build this pipeline. Stating that plainly here rather than implying the full warehouse was modeled.

**Time window:** the starter release's `trend_direction`/`trend_pct` label is a trailing last-30-day vs. prior-30-day comparison; `impressions_prev_30d`/`clicks_prev_30d`/`sessions_prev_30d` are the window immediately before that comparison.

**Excluded, and why:** `content_id`/`client_id` (identifiers, not features); `avg_position`/`ctr`/every 90-day and last-30-day traffic aggregate (window-overlap leakage risk, confirmed by measurement in ML-05, not assumption); `trend_direction`/`trend_pct` (these define the label itself).

In [2]:
print("Missing-data rates for fields actually used as features:")
feature_check_cols = ["word_count","char_count","search_volume","competition","cpc"]
print(df[feature_check_cols].isna().mean().round(3))
print("\nExcluded-for-leakage columns, confirmed absent from the modeling feature set later in this notebook:")
print(["avg_position", "ctr", "impressions_90d", "impressions_last_30d", "trend_direction", "trend_pct"])

Missing-data rates for fields actually used as features:
word_count       0.257
char_count       0.257
search_volume    0.082
competition      0.082
cpc              0.082
dtype: float64

Excluded-for-leakage columns, confirmed absent from the modeling feature set later in this notebook:
['avg_position', 'ctr', 'impressions_90d', 'impressions_last_30d', 'trend_direction', 'trend_pct']


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Label:** `is_declining_label = (trend_direction == 'down')`, an observed trailing-window outcome, not a hand-defined rule (confirmed in ML-03).

**Feature set (leakage-safe):** static/descriptive fields (word count, char count, search volume, competition, CPC, content age, days since last update, content type, main intent, competition level, age tier, freshness tier) plus the one safe traffic window, `*_prev_30d`. Built and locked in ML-05 after a measured leakage audit, not a guess: `_last_30d` columns correlated consistently harder with the label than `_prev_30d` equivalents, and `avg_position`/`ctr` have no safe prev-30d-only version in this release, so both were excluded entirely rather than used with a caveat.

**Baseline:** a plausible, human-writable rule (visible + stale 90+ days + `avg_position` >= 15, gated by impressions), built in ML-07. Re-verified in ML-09 on the exact held-out split used for the final model, not a different, friendlier sample.

**Validation design:** client-grouped split (`GroupShuffleSplit` on `client_id`), zero client overlap confirmed by count, not assumed. ML-05 measured why this matters: a random split scored 0.644 AUC vs. 0.551 for a genuinely grouped split on the same task, evidence of real client-memorization risk on this dataset.

**Leakage checks:** the confession test (deliberately re-adding the label-derived `trend_pct` and watching the score jump toward 1.0, then removing it) was run in ML-05 and reconfirmed in ML-09.

In [3]:
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

static_numeric = ["word_count", "char_count", "search_volume", "competition", "cpc",
                   "content_age_days", "days_since_last_update"]
static_categorical = ["content_type", "main_intent", "competition_level", "age_tier", "freshness_tier"]
safe_traffic = ["impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"]
numeric_cols = static_numeric + safe_traffic
X_numeric = df[numeric_cols].fillna(-1)
X_categorical = pd.get_dummies(df[static_categorical].fillna("unknown"), prefix=static_categorical)
X = pd.concat([X_numeric, X_categorical], axis=1)
y = df["is_declining_label"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df["client_id"]))
Xtr, Xte = X.iloc[train_idx], X.iloc[test_idx]
ytr, yte = y.iloc[train_idx], y.iloc[test_idx]

overlap = len(set(df.iloc[train_idx]["client_id"]) & set(df.iloc[test_idx]["client_id"]))
print(f"client overlap between train/test (must be 0): {overlap}")

# leakage confession test, reconfirmed
Xtr_r, Xte_r, ytr_r, yte_r = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_leaky = X.copy()
X_leaky["trend_pct_LEAKY"] = (df["trend_pct"].fillna(0) - df["trend_pct"].fillna(0).mean()) / (df["trend_pct"].fillna(0).std() + 1e-9)
Xtr_l, Xte_l, ytr_l, yte_l = train_test_split(X_leaky, y, test_size=0.2, random_state=42, stratify=y)

scaler_h, scaler_l = StandardScaler(), StandardScaler()
honest = LogisticRegression(max_iter=2000, random_state=42).fit(scaler_h.fit_transform(Xtr_r), ytr_r)
leaky = LogisticRegression(max_iter=2000, random_state=42).fit(scaler_l.fit_transform(Xtr_l), ytr_l)
auc_honest = roc_auc_score(yte_r, honest.predict_proba(scaler_h.transform(Xte_r))[:, 1])
auc_leaky = roc_auc_score(yte_l, leaky.predict_proba(scaler_l.transform(Xte_l))[:, 1])
print(f"honest AUC: {auc_honest:.3f} | AUC with deliberate leak re-added: {auc_leaky:.3f}")

client overlap between train/test (must be 0): 0


honest AUC: 0.642 | AUC with deliberate leak re-added: 1.000


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

**Model vs. baseline, same split, same metric.** Logistic Regression was chosen over Random Forest because it won on this task, reported as found, not tuned until it looked better (ML-08's real result).

In [4]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

K = 50
test_df = df.iloc[test_idx].copy()

visible = (df["impressions_90d"] >= 500).astype(int)
stale = (df["days_since_last_update"] >= 90).astype(int)
weak_position = ((df["avg_position"] > 0) & (df["avg_position"] >= 15)).astype(int)
baseline_rule_test = (visible.iloc[test_idx] * stale.iloc[test_idx] * weak_position.iloc[test_idx]
                       * df.iloc[test_idx]["impressions_90d"])

scaler = StandardScaler()
lr = LogisticRegression(max_iter=2000, random_state=42).fit(scaler.fit_transform(Xtr), ytr)
rf = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1).fit(Xtr, ytr)

results = pd.DataFrame({
    "method": ["Base rate (guessing)", "Dummy (sort by impressions)", "Rule baseline (ML-07)",
               "Random Forest", "Logistic Regression (final)"],
    f"precision@{K}": [
        yte.mean(),
        precision_at_k(test_df["impressions_90d"].values, yte.values, K),
        precision_at_k(baseline_rule_test.values, yte.values, K),
        precision_at_k(rf.predict_proba(Xte)[:, 1], yte.values, K),
        precision_at_k(lr.predict_proba(scaler.transform(Xte))[:, 1], yte.values, K),
    ],
})
print(results.to_string(index=False))
print("\nChart 1 (docs/img/chart1_results.png) renders this table.")
print("Chart 2 (docs/img/chart2_split.png) shows the same model under a random vs. grouped split.")

                     method  precision@50
       Base rate (guessing)      0.510952
Dummy (sort by impressions)      0.440000
      Rule baseline (ML-07)      0.320000
              Random Forest      0.480000
Logistic Regression (final)      0.600000

Chart 1 (docs/img/chart1_results.png) renders this table.
Chart 2 (docs/img/chart2_split.png) shows the same model under a random vs. grouped split.


## 5. Limitations

*What this work cannot claim.*

**What this work cannot claim:**

- Built and validated on the starter release's client mix and time window; not yet re-validated against the full warehouse or a different season.
- Blind to a page's actual current search ranking and click-through rate by design, `avg_position`/`ctr` are excluded for leakage reasons, a real, known gap, not an oversight.
- Precision@50 = 0.600 describes this exact held-out, client-grouped set. A naive random split on the same data and model reports 0.760, a number close enough to an external reference report's own claimed 0.740 that this paper raises, but does not resolve, a real question about that report's validation design (see Methodology and the discussion below).
- The signal audit (ML-06) found that staleness, the intuitive core assumption behind refresh prioritization, predicts decline only over part of its range and reverses at the extreme (91-180 days: 61.1% decline rate; 181+ days: 47.1%, n=174 and 9,171 respectively). A rule or model leaning on staleness alone will misfire at that extreme.
- Decision-support only. This ranks review priority; it does not diagnose why a page is declining, and it should never be the sole basis for an irreversible content or personnel decision.

## 6. Ranked recommendations

*The action playbook output, the paper's recommendations section.*

The action playbook (ML-10): a ranked queue scored on the exact held-out set validated above, each row with a plain-language reason code and a confidence tier, so an editor sees why, not just a number.

In [5]:
coefs = pd.Series(lr.coef_[0], index=X.columns)
Xte_s_df = pd.DataFrame(scaler.transform(Xte), columns=X.columns, index=Xte.index)
contributions = Xte_s_df * coefs
top_feature = contributions.abs().idxmax(axis=1)

REASON_MAP = {
    "days_since_last_update": "hasn't been refreshed in a while",
    "content_age_days": "one of the older pages in the set",
    "competition": "targets a highly competitive keyword",
    "word_count": "shorter than typical for this content type",
    "impressions_prev_30d": "low prior search visibility",
    "clicks_prev_30d": "low prior click volume",
    "sessions_prev_30d": "low prior session volume",
    "search_volume": "targets a low-search-volume keyword",
    "cpc": "targets a low-commercial-value keyword",
}
def reason_text(feat):
    if feat in REASON_MAP: return REASON_MAP[feat]
    if feat.startswith("age_tier_"): return f"falls in the {feat.replace('age_tier_','')}-day age bracket"
    if feat.startswith("freshness_tier_"): return f"falls in the {feat.replace('freshness_tier_','')}-day freshness bracket"
    if feat.startswith("main_intent_"): return f"{feat.replace('main_intent_','')} intent content"
    if feat.startswith("content_type_"): return f"a {feat.replace('content_type_','')} page"
    if feat.startswith("competition_level_"): return f"{feat.replace('competition_level_','')} competition level"
    return feat

test_df["pred_proba"] = lr.predict_proba(scaler.transform(Xte))[:, 1]
test_df["reason_code"] = top_feature.apply(reason_text).values
test_df["confidence_tier"] = pd.cut(test_df["pred_proba"], bins=[0, 0.4, 0.6, 1.0], labels=["low", "medium", "high"])

os.makedirs("work/outputs", exist_ok=True)
ranked = test_df.sort_values("pred_proba", ascending=False)
ranked[["content_id", "client_id", "pred_proba", "confidence_tier", "reason_code", "is_declining_label"]].to_csv(
    "work/outputs/action_playbook_queue.csv", index=False)
print("written: work/outputs/action_playbook_queue.csv")
print(ranked[["content_id", "pred_proba", "confidence_tier", "reason_code"]].head(10).to_string(index=False))

written: work/outputs/action_playbook_queue.csv
          content_id  pred_proba confidence_tier                              reason_code
content_d0cadc3e2773    0.843278            high falls in the 31-90-day freshness bracket
content_c84a0ab98e90    0.837452            high              low prior search visibility
content_3d6236696109    0.834812            high falls in the 31-90-day freshness bracket
content_f562786a3f25    0.833595            high falls in the 31-90-day freshness bracket
content_ddf7d7516f0f    0.832566            high falls in the 31-90-day freshness bracket
content_4ef69f179958    0.832331            high falls in the 31-90-day freshness bracket
content_e5b7ee456c43    0.832324            high falls in the 31-90-day freshness bracket
content_e906960b0d12    0.830025            high falls in the 31-90-day freshness bracket
content_bf212b1edf1a    0.830002            high falls in the 31-90-day freshness bracket
content_2f51ca262e66    0.826215            high fal

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

Charts the deployed page embeds, generated here so they're reproducible from this notebook, not hand-made separately.

In [6]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

os.makedirs("docs/img", exist_ok=True)
MAIN, TEXT, BG, ACCENT, MUTED, RED = "#1D3557", "#14171C", "#FAFAF8", "#1F7A4D", "#88898F", "#C62828"

# Chart 1: results comparison
fig, ax = plt.subplots(figsize=(7.5, 4.5), facecolor=BG)
ax.set_facecolor(BG)
methods = ["Base rate\n(guessing)", "Dummy\n(sort by traffic)", "Rule baseline\n(ML-07)", "Logistic\nRegression"]
values = list(results[f"precision@{K}"].values[[0, 1, 2, 4]])
bars = ax.bar(methods, values, color=[MUTED, MUTED, RED, ACCENT], width=0.55)
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.015, f"{v:.3f}", ha="center", fontsize=11, color=TEXT, fontweight="bold")
ax.set_ylabel(f"Precision@{K} (held-out, client-grouped split)", fontsize=10, color=TEXT)
ax.set_ylim(0, 0.75)
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(colors=TEXT, labelsize=10)
ax.set_title(f"Precision@{K}: model vs. baselines, same honest split", fontsize=12, color=TEXT, pad=14)
plt.tight_layout()
plt.savefig("docs/img/chart1_results.png", dpi=150, facecolor=BG)
plt.close()
print("saved docs/img/chart1_results.png")

saved docs/img/chart1_results.png


## 8. Five-Minute Demo Outline

*For the Week 8 showcase, optional. Question, method, one chart, one honest result, one recommendation.*

**Question (30 seconds).** FlyRank's editors have thousands of live content pages across 32 clients and only about 50 review slots a week. Which pages should get reviewed first?

**Method (1 minute).** Framed the problem as a ranked queue, not a yes/no classifier. Built a leakage-audited feature set after finding that the two most obviously useful columns, avg_position and ctr, actually leak the label through window overlap, so they were excluded entirely. Validated with a client-grouped split, not a random one, after measuring that a random split lets client memorization inflate the score.

**One chart (1.5 minutes).** Show Figure 2 from the paper: the same model, same data, scores 0.760 precision@50 under a leaky random split and 0.600 under an honest client-grouped split. That 0.160 gap is the whole argument for validating this way, made visible in one chart instead of explained in a paragraph.

**One honest result (1 minute).** The hand-written rule baseline, a genuinely reasonable one, scored 0.320, below the 0.511 base rate. It lost to guessing. That result, not a hunch, is what justified building a model instead of shipping a simpler rule.

**One recommendation (1 minute).** Ship the logistic regression model's ranked queue with reason codes and confidence tiers, not a raw probability list, and treat any prediction built on missing traffic data as lower trust until an editor verifies it by hand.

## 9. Two Shareable Cuts

**Social post (methodology-focused):**

> Spent my FlyRank ML internship capstone finding out my own model's biggest number was partly fake. A random train/test split gave 0.76 precision@50. Same model, same data, split honestly by client instead: 0.60. That 0.16 gap was client memorization, not real signal. Also tested the hand-written rule I would have shipped first: it scored below random guessing. Neither result made it into a clean headline number, and both were more useful than one would have been. Full paper and reproducible notebooks linked below.

**Employer-facing summary (3 sentences):**

> I built a content-refresh prioritization model for FlyRank's editorial teams, ranking which of 30,000 live pages across 32 clients most need review out of a fixed weekly capacity. Using a leakage-audited feature set and a client-grouped validation split, the final model reached 0.600 precision@50 against a 0.511 base rate and a hand-written rule baseline that scored below guessing at 0.320. The full methodology, honest limitations, and a reproducible ranked action queue are documented in a deployed research paper with linked notebooks.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled, markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime, Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/`, then submit your repo URL on the card. Done.